# MCDI502 · Semana 2 · Sumativa 2 — Taller I
## Análisis histórico de los Juegos Olímpicos con Apache Spark

**Grupo:** Grupo 2
**Integrantes:** Verónica Durán Cisterna, Raúl Moya Arriagada, Daniela Rojas Vilches, Manuel Sánchez Cárcamo
**Fecha:** 08-11-2026

**Objetivo:** demostrar RDD, DataFrames, uniones, transformaciones, Spark SQL, agregaciones y particionamiento.

> **Nota de trazabilidad:** El material entregado incluye únicamente `deportista2.csv`, no `deportista.csv`. Para demostrar `union()` sin inventar filas, se divide el archivo disponible en dos RDD complementarios y se reconstruye el total. Esto es una adaptación explícita y **no equivale** a leer los dos archivos solicitados literalmente. Conviene confirmar esta decisión con el docente.

> **Nota de calidad:** En `juegos.json`, el campo `temporada` contiene el **año numérico** y `ciudad` contiene **Verano/Invierno**. Se preservan los campos originales y se crean alias semánticos `anio_juego` y `temporada_juego` para evitar conclusiones equivocadas.


## 1. Configuración del entorno
Ejecutar en Google Colab desde el inicio. Se instala PySpark y se crean variables de entorno, SparkSession y SparkContext. No es necesario descargar manualmente una distribución de Spark.

In [1]:
import os, sys, subprocess
print('Versión de Python:', sys.version.split()[0])
subprocess.run(['java', '-version'], check=True)
%pip -q install pyspark==4.0.5

Versión de Python: 3.13.18


In [2]:
import os, shutil
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark import StorageLevel

java_bin = shutil.which('java')
assert java_bin, 'Java no está disponible en este entorno'
os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(os.path.realpath(java_bin)))
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
spark = (SparkSession.builder.master('local[*]')
         .appName('MCDI502_Taller_I_Juegos_Olimpicos')
         .config('spark.sql.shuffle.partitions', '5')
         .getOrCreate())
sc = spark.sparkContext
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('Spark:', spark.version, '| Contexto:', sc.master)


JAVA_HOME: /usr/lib/jvm/java-21-openjdk-amd64
Spark: 4.0.5 | Contexto: local[*]


## 1.1 Acceso a los datos
* El código detecta el ZIP en `/content`, `/mnt/data` o en `MyDrive` (busca en subcarpetas). Si los cinco archivos ya están descomprimidos en Drive, también los detecta. Los datos no se inventan ni descargan de fuentes externas.

In [ ]:
from pathlib import Path
import zipfile

NECESARIOS = ['deportista2.csv','equipo.csv','evento.csv','resultados.csv','juegos.json']
base_local = Path('/content/mcdi502_datos')
base_local.mkdir(parents=True, exist_ok=True)

# Si se está ejecutando en Colab, montar Drive solo cuando haga falta.
if Path('/content').exists():
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
    except ImportError:
        pass

carpetas_busqueda = [Path('/content'), Path('/mnt/data'), Path('/content/drive/MyDrive')]

def localizar_archivo(nombre):
    for carpeta in carpetas_busqueda:
        if carpeta.exists():
            directo = carpeta / nombre
            if directo.is_file():
                return directo
            encontrados = list(carpeta.rglob(nombre))
            if encontrados:
                return encontrados[0]
    return None

zip_datos = localizar_archivo('mcdi502_s2_datos.zip')
if zip_datos:
    with zipfile.ZipFile(zip_datos) as z:
        z.extractall(base_local)
    print('ZIP extraído:', zip_datos)
else:
    print('ZIP no encontrado; se buscarán los archivos individuales.')

rutas = {}
for nombre in NECESARIOS:
    ruta = base_local / nombre
    if not ruta.exists():
        ruta = localizar_archivo(nombre)
    assert ruta is not None and ruta.is_file(), f'Falta {nombre}. Sube el ZIP adjunto o el archivo individual.'
    rutas[nombre] = str(ruta)
    print(nombre, '→', ruta)


Mounted at /content/drive


## 2. RDD: creación y unión
El archivo `deportista2.csv` no tiene cabecera. Se asignan los siete campos solicitados. Se divide el archivo en dos subconjuntos disjuntos según el ID (par/impar) para demostrar `union()` sin duplicar filas. El RDD `deportista` queda con **6 particiones**.

In [ ]:
# El CSV se lee como líneas de texto con seis particiones.
rdd_archivo = sc.textFile(rutas['deportista2.csv'], minPartitions=6)

# Se parsea el CSV de forma segura, respetando nombres que pudieran contener comas entre comillas.
import csv

def parsear_deportista(linea):
    fila = next(csv.reader([linea]))
    if len(fila) != 7:
        raise ValueError(f'Fila de deportista con {len(fila)} columnas: {fila}')
    return fila

rdd_parseado = rdd_archivo.map(parsear_deportista)
# Dividir en dos mitades sin solapamiento; no representa dos fuentes independientes.
rdd_deportista = rdd_parseado.filter(lambda x: int(x[0]) % 2 == 0).repartition(6)
rdd_deportista2 = rdd_parseado.filter(lambda x: int(x[0]) % 2 != 0)
deportistaTotal = rdd_deportista.union(rdd_deportista2)
print('Particiones RDD deportista:', rdd_deportista.getNumPartitions())
print('Registros fuente:', rdd_parseado.count())
print('Registros RDD deportista:', rdd_deportista.count())
print('Registros RDD deportista2:', rdd_deportista2.count())
print('Registros deportistaTotal:', deportistaTotal.count())
assert deportistaTotal.count() == rdd_parseado.count(), 'La unión no conserva todos los registros'


In [ ]:
esquema_deportista = T.StructType([
    T.StructField('deportista_id', T.IntegerType(), False),
    T.StructField('nombre', T.StringType(), True),
    T.StructField('genero', T.IntegerType(), True),
    T.StructField('edad', T.IntegerType(), True),
    T.StructField('altura', T.DoubleType(), True),
    T.StructField('peso', T.DoubleType(), True),
    T.StructField('equipo_id', T.IntegerType(), True),
])

def entero(x):
    try: return int(x) if x not in ('', 'NA', 'null') else None
    except (ValueError, TypeError): return None

def decimal(x):
    try: return float(x) if x not in ('', 'NA', 'null') else None
    except (ValueError, TypeError): return None

rdd_tipos = deportistaTotal.map(lambda x: (
    entero(x[0]), x[1], entero(x[2]), entero(x[3]),
    decimal(x[4]), decimal(x[5]), entero(x[6])
))
deportista = spark.createDataFrame(rdd_tipos, schema=esquema_deportista)
deportista.printSchema()
deportista.show(5, truncate=False)


## 3. Transformaciones RDD
Mayores de edad: `edad >= 18`; mujeres: `genero == 2`. Las conversiones a mayúsculas se hacen sobre los campos de texto, preservando números.

In [ ]:
MayorEdad = deportistaTotal.filter(lambda x: entero(x[3]) is not None and entero(x[3]) >= 18)
Deportistas_mujer = deportistaTotal.filter(lambda x: entero(x[2]) == 2)
Deportistas_mayusculas = deportistaTotal.map(lambda x: [campo.upper() for campo in x])
print('Mayores de edad:', MayorEdad.count())
print('Mujeres:', Deportistas_mujer.count())
print('Ejemplos de mayores:', MayorEdad.take(3))
print('Ejemplos de mujeres:', Deportistas_mujer.take(3))
print('Ejemplos en mayúsculas:', Deportistas_mayusculas.take(3))


## 4. DataFrames: carga, optimización e integración
Los nombres reales de columnas son distintos a los de la guía. Se renombran las claves para integrar sin ambigüedades. `resultados.csv` usa `;` y el valor `NA` es una categoría de **sin medalla**, no una medalla.

In [ ]:
Equipo = (spark.read.option('header', True).option('inferSchema', True)
          .csv(rutas['equipo.csv'])
          .withColumnRenamed('id','equipo_id'))
Evento = (spark.read.option('header', True).option('inferSchema', True)
          .csv(rutas['evento.csv'])
          .withColumnRenamed('cod_evento','evento_id'))
Resultado = (spark.read.option('header', True).option('inferSchema', True)
             .option('sep',';').option('nullValue','__NULO_INEXISTENTE__')
             .csv(rutas['resultados.csv']))
Juego = spark.read.option('multiLine',True).json(rutas['juegos.json'])

# Ajustes de tipos en las claves para joins fiables.
Equipo = Equipo.withColumn('equipo_id',F.col('equipo_id').cast('int'))
Evento = Evento.withColumn('evento_id',F.col('evento_id').cast('int'))
Resultado = (Resultado.withColumn('deportista_id',F.col('deportista_id').cast('int'))
             .withColumn('evento_id',F.col('evento_id').cast('int'))
             .withColumn('juego_id',F.col('juego_id').cast('int')))
Juego = Juego.withColumn('juego_id',F.col('juego_id').cast('int'))

for nombre,tabla in [('Deportista',deportista),('Equipo',Equipo),('Evento',Evento),('Resultado',Resultado),('Juego',Juego)]:
    print('\n',nombre,'filas:',tabla.count())
    tabla.printSchema()
    tabla.show(3,truncate=False)


In [ ]:
# Optimización razonada: tablas de dimensión pequeñas se difunden mediante broadcast.
# Se usa cache en las dimensiones que se reutilizan; la tabla grande de resultados
# se mantiene sin cachear por defecto para evitar uso innecesario de memoria.
Equipo = Equipo.cache()
Evento = Evento.cache()
Juego = Juego.cache()
for tabla in [Equipo,Evento,Juego]:
    tabla.count()  # materializa el cache
print('Dimensiones pequeñas almacenadas en caché.')


### 4.1 Calidad y cobertura de la integración
`resultados.csv` contiene registros de deportistas que pueden no estar en el único archivo disponible. Se conservan **todos los resultados** con `left join` y se deja explícita la cobertura. Las estadísticas que requieren edad, sexo, altura o peso solo pueden calcularse sobre deportistas identificados.

In [ ]:
# Comprobar integridad de IDs antes de unir.
print('Deportistas disponibles:',deportista.select('deportista_id').distinct().count())
print('Resultados históricos:',Resultado.count())
print('Resultados sin deportista disponible:',
      Resultado.join(deportista.select('deportista_id'),'deportista_id','left_anti').count())

# Resultado es la tabla de hechos: preservamos su cantidad de filas.
Integrado = (Resultado.alias('r')
    .join(deportista.alias('d'), 'deportista_id','left')
    .join(F.broadcast(Equipo).alias('e'), 'equipo_id','left')
    .join(F.broadcast(Evento).alias('ev'), 'evento_id','left')
    .join(F.broadcast(Juego).alias('j'), 'juego_id','left'))

# El JSON viene con etiquetas semánticas intercambiadas: año numérico y estación.
Integrado = (Integrado
    .withColumn('anio_juego',F.col('temporada').cast('int'))
    .withColumn('temporada_juego',F.col('ciudad')))
print('Registros integrados:',Integrado.count())
assert Integrado.count() == Resultado.count(), 'Revisar duplicación de claves en las tablas de dimensión'
Integrado.select('deportista_id','nombre','equipo','evento','medalla','anio_juego','temporada_juego').show(10,truncate=False)


## 5. Paralelismo: cinco particiones

In [ ]:
print('Particiones antes:', Integrado.rdd.getNumPartitions())
Integrado = Integrado.repartition(5)
print('Particiones después:', Integrado.rdd.getNumPartitions())
assert Integrado.rdd.getNumPartitions() == 5


## 6. Inspección del DataFrame integrado

In [ ]:
print('Cantidad de filas:',Integrado.count())
print('Tipos de datos:',Integrado.dtypes)
Integrado.printSchema()
Integrado.show(5,truncate=False)


## 7. Columnas calculadas
IMC = peso (kg) / altura (m)². Se protegen los valores nulos y alturas cero. `Descripción_sexo` distingue 1=Hombre y 2=Mujer.

In [ ]:
Integrado = (Integrado
    .withColumn('IMC', F.when((F.col('altura') > 0) & F.col('peso').isNotNull(),
              F.round(F.col('peso') / F.pow(F.col('altura')/F.lit(100.0),2),2)))
    .withColumn('Descripción_sexo',
        F.when(F.col('genero')==1,F.lit('Hombre'))
         .when(F.col('genero')==2,F.lit('Mujer'))
         .otherwise(F.lit('No informado'))))
Integrado.select('nombre','altura','peso','IMC','genero','Descripción_sexo').show(12,truncate=False)


## 8. Agregaciones solicitadas
Las medallas se cuentan por registro de resultado, no por medallas físicas de equipos. El análisis por edad/altura utiliza solo los registros con deportista presente. Se explicita la cobertura.

In [ ]:
# 8.1 Cantidad de medallas por equipo
medallas_equipo = (Integrado.filter(F.col('medalla').isin('Gold','Silver','Bronze'))
    .groupBy('equipo')
    .agg(F.sum(F.when(F.col('medalla')=='Gold',1).otherwise(0)).alias('oro'),
         F.sum(F.when(F.col('medalla')=='Silver',1).otherwise(0)).alias('plata'),
         F.sum(F.when(F.col('medalla')=='Bronze',1).otherwise(0)).alias('bronce'))
    .orderBy(F.desc('oro'),F.desc('plata'),F.desc('bronce')))
medallas_equipo.show(20,truncate=False)


In [ ]:
# 8.2 Edad por tipo de medalla (solo medallas reales)
edad_medalla = (Integrado.filter(F.col('medalla').isin('Gold','Silver','Bronze'))
    .groupBy('medalla')
    .agg(F.count('edad').alias('observaciones_con_edad'),
         F.sum('edad').alias('suma_edad'),
         F.round(F.avg('edad'),2).alias('promedio_edad'),
         F.max('edad').alias('maximo_edad'),
         F.min('edad').alias('minimo_edad')))
edad_medalla.show(truncate=False)


In [ ]:
# 8.3 Altura por temporada: el archivo JSON codifica Verano/Invierno en 'ciudad'.
temporada = (Integrado.groupBy('temporada_juego')
    .agg(F.count('altura').alias('observaciones_con_altura'),
         F.sum('altura').alias('suma_altura'),
         F.round(F.avg('altura'),2).alias('promedio_altura'),
         F.max('altura').alias('maximo_altura'),
         F.min('altura').alias('minimo_altura'))
    .orderBy('temporada_juego'))
temporada.show(truncate=False)


In [ ]:
# 8.4 Edad por sexo
sexo = (Integrado.groupBy('Descripción_sexo')
    .agg(F.count('edad').alias('observaciones_con_edad'),
         F.sum('edad').alias('suma_edad'),
         F.round(F.avg('edad'),2).alias('promedio_edad'),
         F.max('edad').alias('maximo_edad'),
         F.min('edad').alias('minimo_edad'))
    .orderBy('Descripción_sexo'))
sexo.show(truncate=False)


## 9. Spark SQL: evidencia complementaria del indicador 3.3
Se crean vistas temporales y se replica una agregación con SQL. Esto permite comparar la API DataFrame con Spark SQL.

In [ ]:
Integrado.createOrReplaceTempView('olimpicos')
consulta_sql = spark.sql("""
SELECT equipo,
       SUM(CASE WHEN medalla = 'Gold' THEN 1 ELSE 0 END) AS oro,
       SUM(CASE WHEN medalla = 'Silver' THEN 1 ELSE 0 END) AS plata,
       SUM(CASE WHEN medalla = 'Bronze' THEN 1 ELSE 0 END) AS bronce
FROM olimpicos
WHERE medalla IN ('Gold','Silver','Bronze') AND equipo IS NOT NULL
GROUP BY equipo
ORDER BY oro DESC, plata DESC, bronce DESC
""")
consulta_sql.show(15,truncate=False)


## 10. Validaciones finales y análisis de datos faltantes
Las filas sin deportista disponible no pueden recibir edad, altura, peso, sexo o equipo de forma fiable. La integración no las descarta, pero las agregaciones por deportista tienen cobertura parcial.

In [ ]:
resumen_cobertura = Integrado.agg(
    F.count('*').alias('resultados_totales'),
    F.count('nombre').alias('resultados_con_deportista'),
    F.count('edad').alias('resultados_con_edad'),
    F.count('altura').alias('resultados_con_altura'),
    F.count('equipo').alias('resultados_con_equipo'))
resumen_cobertura.show(truncate=False)
print('Verificación particiones:',Integrado.rdd.getNumPartitions())
print('Medallas identificadas:',Integrado.groupBy('medalla').count().orderBy('medalla').collect())


## 11. Reflexión técnica para el informe y video

**Completar después de ejecutar:**

1. **RDD y unión:** explicar por qué se usaron seis particiones y cómo se conservó el total de filas al unir dos subconjuntos del único archivo recibido.
2. **Datos incompletos:** informar el porcentaje de resultados que se vincularon con deportistas y explicar por qué las métricas de edad, sexo, altura y equipo no representan toda la base histórica.
3. **Calidad semántica:** explicar la inconsistencia de `juegos.json` y el uso de `temporada_juego`.
4. **Optimización:** justificar `broadcast`, `cache` y `repartition(5)`; no afirmar una mejora de velocidad sin mediciones.
5. **Agregaciones:** comentar diferencias observadas en medallas por equipo, edad por medalla, altura por temporada y edad por sexo, usando cifras de las salidas.

